# Quantization Aware Training + Knowledge Distillation using Export Model

In [1]:
import os
import time
import copy
import numpy as np
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import DataLoader
from typing import Callable, Optional, Any

from src.utils import load_data, process_callbacks
from src.Quantization.quantization_utils.model_setup import qat_kd_setup
from src.utils import compute_loss_and_predictions, calculate_metrics, plot_train_val_curve, test_inference
from src.Quantization.quantization_utils.conversions.litert import quantize_pytorch_model, convert_pytorch_model_to_tflite, check_quantized_modules
from src.utils import benchmark

2025-04-10 14:05:24.160297: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-10 14:05:24.225439: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744308324.250331    6957 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744308324.260173    6957 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744308324.311485    6957 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
# Specify random seed for repeatable results
_ = torch.manual_seed(191009)

logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Avoid adding duplicate handlers
if not logger.handlers:
    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)
    stream_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(stream_handler)

### Helper Functions

In [3]:

def compute_distillation_loss(teacher_logits: torch.Tensor, student_logits: torch.Tensor, T: float) -> torch.Tensor:
    """
    Computes the distillation loss using KL divergence with softened logits.
    
    The teacher and student logits are scaled by the temperature T, and the KL divergence 
    is computed with 'batchmean' reduction. The loss is scaled by T^2 as recommended in 
    distillation literature.
    
    Parameters:
        teacher_logits (torch.Tensor): Logits from the teacher model.
        student_logits (torch.Tensor): Logits from the student model.
        T (float): Temperature to soften the logits.
    
    Returns:
        torch.Tensor: The computed distillation loss.
    """
    # Compute KL divergence loss between softened logits.
    soft_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction='batchmean'
    ) * (T ** 2)
    return soft_loss

### Training QAT + KD

In [4]:
def train_knowledge_distillation(
    teacher: nn.Module,
    student: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    learning_rate: float,
    epochs: int,
    criterion: Callable[[torch.Tensor, torch.Tensor], tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    optimizer: torch.optim.Optimizer,
    callbacks: list[Any],
    quant_mode: str,
    T: float,
    soft_target_loss_weight: float,
    ce_loss_weight: float,
    device: torch.device
) -> dict[str, list[float]]:
    """
    Trains the student model using knowledge distillation from the teacher model.
    
    The total loss is a weighted sum of:
      - The distillation loss computed as KL divergence between softened teacher and student logits.
      - The standard loss computed on true labels.
      
    During validation, the student model is switched to evaluation mode (and optionally quantized) 
    to simulate the behavior of the final quantized model. Training and validation metrics are logged 
    and stored per epoch.
    
    Parameters:
        teacher (nn.Module): The pre-trained teacher model.
        student (nn.Module): The student model to be trained.
        train_loader (DataLoader): DataLoader for the training dataset.
        val_loader (DataLoader): DataLoader for the validation dataset.
        learning_rate (float): The learning rate used (for logging purposes).
        epochs (int): Number of training epochs.
        criterion (Callable): Loss function for true label loss (e.g., nn.CrossEntropyLoss). 
                              This function should return a tuple: (label_loss, probabilities, predictions).
        optimizer (torch.optim.Optimizer): Optimizer for updating the student model parameters.
        callbacks (list[Any]): List of callback objects with methods on_epoch_start and on_epoch_end.
        quant_mode (str): Quantization mode string (e.g., 'eager', 'fx', or 'export') used during validation.
        T (float): Temperature for softening the logits.
        soft_target_loss_weight (float): Weight for the distillation (soft target) loss.
        ce_loss_weight (float): Weight for the cross entropy loss computed on true labels.
        device (torch.device): Device to perform training on.
    
    Returns:
        dict[str, list[float]]: A dictionary containing training and validation loss history 
                                  with keys 'train' and 'val'.
    """
    loss_dict = {'train': [], 'val': []}
    # Set teacher to evaluation mode (its parameters won't be updated)
    teacher.eval()
    
    for epoch in range(epochs):
        logs = {"model": student, 
                "batch_size": train_loader.batch_size, 
                "learning_rate": learning_rate, 
                "optimizer": optimizer,
                "criterion": criterion,
                "epoch": epoch}

        # Trigger on_epoch_start callbacks
        for callback in callbacks:
            callback.on_epoch_start(epoch, logs)

        for phase in ['train', 'val']:
            is_train = phase == 'train'

            # For training phase, we use the original student model.
            # For validation, we create a deep copy and quantize it.
            if is_train:
                # current_student = student.to(device)
                # Ensure the model is in train mode
                torch.ao.quantization.move_exported_model_to_train(student)
            else:
                # Create a deep copy to avoid altering the training model
                # current_student = copy.deepcopy(student).to(device)
                # Quantize the copied student model for validation
                # Pass save_dir as None to avoid saving the state dict
                # current_student = quantize_pytorch_model(current_student, quant_mode=quant_mode, save_dir=None)
                # Switch to evaluation mode to perform inference
                torch.ao.quantization.move_exported_model_to_eval(student)

            data_loader = train_loader if is_train else val_loader

            running_loss = 0.0
            total_samples  = 0.0
            predictions, ground_truths, probabilities = [], [], []

            with tqdm(total=len(data_loader), desc=f"{phase.capitalize()} Epoch {epoch + 1}/{epochs}") as pbar:
                for inputs, labels in data_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    
                    # Zero gradients only during training
                    if is_train:
                        optimizer.zero_grad()
                    
                    # Teacher forward pass without gradient computation.
                    with torch.no_grad():
                        teacher = teacher.to(device)
                        teacher_logits = teacher(inputs)
                    
                    # Enable gradients only in training phase
                    with torch.set_grad_enabled(is_train):                        
                        student = student.to(device)
                        student_logits = student(inputs)

                        # Compute soft targets loss using KL divergence.
                        soft_loss = compute_distillation_loss(teacher_logits=teacher_logits, student_logits=student_logits, T=T)

                        # Compute the true label loss and obtain predictions/probabilities
                        label_loss, probs, preds = compute_loss_and_predictions(student_logits, labels, criterion)
                        
                        # Combine the losses using the specified weights.
                        loss = soft_target_loss_weight * soft_loss + ce_loss_weight * label_loss

                        # Backpropagate and update weights if in training mode.
                        if is_train:
                            loss.backward()
                            optimizer.step()
                    
                    # Update running loss and sample count
                    batch_size = inputs.size(0)
                    running_loss += loss.item() * batch_size
                    total_samples += batch_size

                    # Accumulate predictions and ground truths for metrics
                    ground_truths.extend(labels.cpu().detach().numpy())
                    predictions.extend(preds.cpu().detach().numpy())
                    probabilities.extend(probs.cpu().detach().numpy())
                    
                    # Update progress bar with average loss so far
                    pbar.set_postfix(loss=f"{running_loss / total_samples:.4f}")
                    pbar.update(1)
                
            # Compute aggregated metrics after epoch
            epoch_loss = running_loss / total_samples
            
            y_true = np.array(ground_truths)
            y_pred = np.array(predictions)
            y_proba = np.array(probabilities) if probabilities else None

            # Calculate metrics
            metrics = calculate_metrics(y_true, y_pred, y_proba)

            loss_dict[phase].append(epoch_loss)
            logging.info(f"{phase.capitalize()} Loss: {epoch_loss:.4f}")
            logging.info(f"{phase.capitalize()} Metrics: " + ", ".join([f"{key}: {value:.4g}" for key, value in metrics.items()]))
            logs.update({f"{phase}_loss": epoch_loss, **metrics})

        # Trigger on_epoch_end callbacks
        for callback in callbacks:
            callback.on_epoch_end(epoch, logs)

        # Check for early stopping
        if any(getattr(cb, "early_stop", False) for cb in callbacks):
            break

    return loss_dict

### Training Loop

In [5]:
def train_qat_kd(
    teacher_name: str,
    student_name: str,
    data_loaders: dict[str, DataLoader],
    save_dir: str,
    learning_rate: float = 0.001,
    epochs: int = 5,
    criterion: str = "cross_entropy",
    optimizer: str = "adam",
    callbacks: Optional[list[Any]] = None,
    teacher_model_weights: Optional[str] = None,
    quant_mode: str = "export",
    config: str = "qnnpack",
    class_weights: bool = False,
    device: torch.device = torch.device("cuda")
) -> tuple[nn.Module, nn.Module]:
    """
    Trains a student model using Knowledge Distillation (KD) combined with Quantization-Aware Training (QAT).
    
    This function performs the following steps:
      1. Loads the teacher and student models using their respective model names. The student model is loaded
         as a standard model and then prepared for QAT via quantization_mode.
      2. Configures the loss function and optimizer using helper functions.
      3. Runs the training loop for a specified number of epochs while applying knowledge distillation 
         (combining soft targets from the teacher with the true label loss).
      4. During training, checkpoints and callbacks are handled. After training, the best model weights 
         are loaded and the student model is quantized for export.
      5. If a test DataLoader is provided, the quantized student model is evaluated.
    
    Parameters:
        teacher_name (str): Name of the pre-trained teacher model (e.g., 'resnet50', 'efficientnet_b0').
        student_name (str): Name of the student model to be trained with QAT.
        data_loaders (dict[str, DataLoader]): Dictionary containing DataLoaders for dataset splits 
                                              (keys: 'train', 'val', and optionally 'test').
        save_dir (str): Directory where the trained model, checkpoints, and metrics will be saved.
        learning_rate (float, optional): Learning rate for the optimizer. Defaults to 0.001.
        epochs (int, optional): Number of training epochs. Defaults to 5.
        criterion (str, optional): Loss function name (e.g., 'cross_entropy', 'bce'). Defaults to 'cross_entropy'.
        optimizer (str, optional): Optimizer name (e.g., 'adam', 'sgd'). Defaults to 'adam'.
        callbacks (Optional[list[Any]], optional): List of callback objects for monitoring, checkpointing,
                                                     and early stopping. Defaults to None.
        teacher_model_weights (Optional[str], optional): Path to pre-trained weights for the teacher model.
                                                         Defaults to None.
        quant_mode (str, optional): Quantization mode to use for preparing the student model 
                                    ('eager', 'fx', or 'export'). Defaults to 'export'.
        config (str, optional): QAT configuration identifier ('qnnpack' for default QAT config for qnnpack 
                                or custom). Defaults to 'qnnpack'.
        class_weights (bool, optional): Whether to compute and apply class weights for imbalanced datasets.
                                        Defaults to False.
        device (torch.device, optional): Device to perform operations on. Defaults to torch.device("cuda").
    
    Returns:
        tuple[nn.Module, nn.Module]: A tuple containing:
            - teacher_model: The loaded teacher model.
            - quantized_student: The quantized student model after training.
    
    Notes:
        - The student model is prepared for QAT using example inputs from the training DataLoader.
        - After training, the best model weights are loaded from a checkpoint, and the student model is quantized 
          (using quantize_pytorch_model) for export.
        - Training metrics (loss curves and evaluation metrics) are saved to the specified directory.
    """
    # Load dataset
    logging.info("Loading datasets...")
    train_loader = data_loaders["train"]
    val_loader = data_loaders["val"]
    logging.info("Datasets loaded successfully.")
    
    # Set up teacher and student models, loss function, and optimizer.
    teacher, student, criterion_fn, optimizer_obj = qat_kd_setup(teacher=teacher_name, 
                                                                student=student_name,
                                                                learning_rate=learning_rate, 
                                                                criterion=criterion, 
                                                                optimizer=optimizer, 
                                                                teacher_model_weights=teacher_model_weights, 
                                                                dataloader=train_loader, 
                                                                quant_mode=quant_mode,
                                                                config=config,
                                                                class_weights=class_weights,
                                                                device=torch.device("cpu"))
    logging.info("Model setup complete.")

    # Set up checkpoint and metrics directories.
    quantized_model_path = os.path.join(save_dir, f"{student_name}_qat_kd.pth")
    metrics_save_dir = os.path.join(save_dir, "metrics")
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(metrics_save_dir, exist_ok=True)
    logging.info(f"Checkpoints and metrics will be saved to: {save_dir}")

    # Trigger training start callbacks.
    if callbacks is not None:
        for callback in callbacks:
            callback.on_train_start(logs={})

    # Training loop
    logging.info("Starting training...")
    start_time = time.time()

    # Train the student model using knowledge distillation.
    loss_dict = train_knowledge_distillation(
                            teacher=teacher, 
                            student=student, 
                            train_loader=train_loader,
                            val_loader=val_loader,
                            learning_rate=learning_rate,
                            epochs=epochs, 
                            criterion=criterion_fn,
                            optimizer=optimizer_obj,
                            callbacks=callbacks,
                            quant_mode=quant_mode,
                            T=2, 
                            soft_target_loss_weight=0.25, 
                            ce_loss_weight=0.75, 
                            device=device
                        )
    elapsed_time = time.time() - start_time
    logging.info(f"Training complete in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")

    # Trigger training end callbacks.
    if callbacks is not None:
        for callback in callbacks:
            callback.on_train_end(logs={"model": student, "optimizer": optimizer_obj})

    # Load the best student model weights from checkpoint.
    model_data = torch.load(quantized_model_path, weights_only=True)
    student.load_state_dict(model_data)
    logging.info(f"Best model weights loaded from: {quantized_model_path}")


    # Quantize the student model for export.
    quantized_student = quantize_pytorch_model(student.to("cpu"), quant_mode, save_dir=os.path.join(save_dir, "quantized_state.pth"))
    # Plot training and validation loss curves.
    plot_train_val_curve(loss_dict, save_path=os.path.join(metrics_save_dir, "loss_curve.png"))

    # If a test set is provided, perform inference evaluation on the quantized student model.
    if "test" in data_loaders and data_loaders["test"] is not None:
        test_inference(quantized_student, data_loaders["test"], torch.device("cpu"), metrics_save_dir)

    return teacher, quantized_student

### Hyperparameters

In [ ]:
dataset = "SkinCancer"
batch_size = 32
learning_rate = 0.001
epochs = 1
save_dir = f"models/{dataset}/Quantized"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher_model = "mobilenet_v2"
student_model = "mobilenet_v2"
args = {"callbacks": ["ModelCheckpoint", "EarlyStopping", "ReduceLROnPlateau"], 
        "save_dir": save_dir, 
        "model_name": student_model,
        "save_name": "qat_kd"}
CALLBACKS = list(process_callbacks(args).values())

os.makedirs(save_dir, exist_ok=True)

dataloaders = load_data(dataset=dataset, batch_size=batch_size)

teacher_model_weights = 'models/SkinCancer/mobilenet_v2_best_model.pth'  # Update with your saved model weights

criterion = "cross_entropy"
optimizer = "adam"
# Before any quantization setup
# torch.backends.quantized.engine = 'qnnpack'
# TODO: Organize repo
teacher, quantized_student = train_qat_kd(
                                        teacher_name=teacher_model, 
                                        student_name=student_model,
                                        data_loaders=dataloaders,
                                        save_dir=save_dir,
                                        learning_rate=learning_rate,
                                        epochs=epochs,
                                        criterion = criterion,
                                        optimizer = optimizer,
                                        callbacks=CALLBACKS,
                                        quant_mode="export",
                                        teacher_model_weights=teacher_model_weights,
                                        device=device,
                                )


INFO:root:Train dataset size: 29322
INFO:root:Class distribution for train dataset:
INFO:root:  Class 'Actinic keratoses': 693 samples
INFO:root:  Class 'Basal cell carcinoma': 2658 samples
INFO:root:  Class 'Benign keratosis-like lesions': 2099 samples
INFO:root:  Class 'Chickenpox': 900 samples
INFO:root:  Class 'Cowpox': 792 samples
INFO:root:  Class 'Dermatofibroma': 191 samples
INFO:root:  Class 'HFMD': 1932 samples
INFO:root:  Class 'Healthy': 1368 samples
INFO:root:  Class 'Measles': 660 samples
INFO:root:  Class 'Melanocytic nevi': 10300 samples
INFO:root:  Class 'Melanoma': 3617 samples
INFO:root:  Class 'Monkeypox': 3408 samples
INFO:root:  Class 'Squamous cell carcinoma': 502 samples
INFO:root:  Class 'Vascular lesions': 202 samples
INFO:root:Val dataset size: 3660
INFO:root:Class distribution for val dataset:
INFO:root:  Class 'Actinic keratoses': 86 samples
INFO:root:  Class 'Basal cell carcinoma': 332 samples
INFO:root:  Class 'Benign keratosis-like lesions': 262 samples


Model prepared using Export Mode QAT.


Train Epoch 1/1:  27%|██▋       | 245/917 [00:52<02:27,  4.55it/s, loss=1.1223]

In [ ]:
benchmark(model1=teacher.to("cpu"), model2=quantized_student.to("cpu"), dataloader=dataloaders["test"], device=torch.device("cpu"))

INFO:root:Average inference time per sample: 0.000730 seconds
INFO:root:Throughput: 1369.72 samples/second
INFO:root:Average inference time per sample: 0.001998 seconds
INFO:root:Throughput: 500.60 samples/second
INFO:root:Time speedup: 0.37x
INFO:root:Throughput speedup: 0.37x
INFO:root:Process memory usage (CPU): 4887.91 MB
INFO:root:Process memory usage (CPU): 4887.91 MB
INFO:root:Average CPU idle power consumption (pyRAPL): 87.91 Watts


Label : inference
Begin : Sat Apr  5 19:37:06 2025
Duration : 5000046.1800 us
-------------------------------
PKG :
	socket 0 :  439584750.0000 uJ


INFO:root:Average CPU power consumption: 117.67 Watts


Label : inference
Begin : Sat Apr  5 19:37:12 2025
Duration : 3309434.4410 us
-------------------------------
PKG :
	socket 0 :  389452471.0000 uJ


INFO:root:Average CPU power consumption: 125.46 Watts


Label : inference
Begin : Sat Apr  5 19:37:17 2025
Duration : 3885990.6260 us
-------------------------------
PKG :
	socket 0 :  487559976.0000 uJ


INFO:root:Latency percentiles: P50=0.052746s, P95=0.053539s, P99=0.061333s
INFO:root:Latency percentiles: P50=0.147367s, P95=0.164274s, P99=0.166594s
INFO:root:Throughput per Watt: 11.64 samples/sec/Watt
INFO:root:Throughput per Watt: 3.99 samples/sec/Watt


                             Metric  Model 1  Model 2
                    Model Size (MB)   9.1899  11.1504
        Inference Time (sec/sample) 0.000730 0.001998
           Throughput (samples/sec)  1369.72   500.60
                  Memory Usage (MB)  4887.91  4887.91
                 Idle Power (Watts)    87.91        -
                  Avg Power (Watts)   117.67   125.46
         Energy per Sample (Joules) 0.085910 0.250618
Throughput per Watt (samples/sec/W)    11.64     3.99
                  Latency P50 (sec) 0.052746 0.147367
                  Latency P95 (sec) 0.053539 0.164274
                  Latency P99 (sec) 0.061333 0.166594
                       Time Speedup    0.37x        -
                 Throughput Speedup    0.37x        -


### Final Quantization for model (Turn Fake-Quant tensors into real quant tensors)

In [ ]:
quant_copy = copy.deepcopy(quantized_student).to("cpu")

In [ ]:
print(quant_copy)

GraphModule(
  (features): Module(
    (0): Module(
      (0): Module()
    )
    (1): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module()
      )
    )
    (2): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (3): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (4): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (5): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (6): Module(
      (conv): Module(
        (0): 

In [ ]:
quant_copy.print_readable()

In [ ]:
check_quantized_modules(quant_copy)

INFO:root:: <class 'torch.fx.graph_module.GraphModule.__new__.<locals>.GraphModuleImpl'>
INFO:root:features: <class 'torch.nn.modules.module.Module'>
INFO:root:features.0: <class 'torch.nn.modules.module.Module'>
INFO:root:features.0.0: <class 'torch.nn.modules.module.Module'>
INFO:root:features.1: <class 'torch.nn.modules.module.Module'>
INFO:root:features.1.conv: <class 'torch.nn.modules.module.Module'>
INFO:root:features.1.conv.0: <class 'torch.nn.modules.module.Module'>
INFO:root:features.1.conv.0.0: <class 'torch.nn.modules.module.Module'>
INFO:root:features.1.conv.1: <class 'torch.nn.modules.module.Module'>
INFO:root:features.2: <class 'torch.nn.modules.module.Module'>
INFO:root:features.2.conv: <class 'torch.nn.modules.module.Module'>
INFO:root:features.2.conv.0: <class 'torch.nn.modules.module.Module'>
INFO:root:features.2.conv.0.0: <class 'torch.nn.modules.module.Module'>
INFO:root:features.2.conv.1: <class 'torch.nn.modules.module.Module'>
INFO:root:features.2.conv.1.0: <clas

In [ ]:
quant_copy.eval()

GraphModule(
  (features): Module(
    (0): Module(
      (0): Module()
    )
    (1): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module()
      )
    )
    (2): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (3): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (4): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (5): Module(
      (conv): Module(
        (0): Module(
          (0): Module()
        )
        (1): Module(
          (0): Module()
        )
        (2): Module()
      )
    )
    (6): Module(
      (conv): Module(
        (0): 

In [16]:
example_inputs = next(iter(dataloaders["train"]))[0].to("cpu")
torch.ao.quantization.move_exported_model_to_eval(quantized_student)
convert_pytorch_model_to_tflite(quantized_student, os.path.join(save_dir, f"{student_model}.tflite"), (example_inputs,))

INFO:root:Converting PyTorch model to TensorFlow Lite format.


INFO:tensorflow:Assets written to: /tmp/tmph2v_j_3_/assets


INFO:tensorflow:Assets written to: /tmp/tmph2v_j_3_/assets
W0000 00:00:1744136961.069521    8871 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1744136961.069540    8871 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2025-04-08 14:29:21.069666: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmph2v_j_3_
2025-04-08 14:29:21.070610: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-04-08 14:29:21.070615: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmph2v_j_3_
2025-04-08 14:29:21.079586: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-04-08 14:29:21.148474: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmph2v_j_3_
2025-04-08 14:29:21.164894: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 95232 

In [17]:
print("qat_kd: ", os.path.getsize("models/SkinCancer/Quantized/mobilenet_v2_qat_kd.pth") / 1e6)
print("quantized_state: ", os.path.getsize("models/SkinCancer/Quantized/quantized_state.pth") / 1e6)
print("TFlite:", os.path.getsize("models/SkinCancer/Quantized/mobilenet_v2.tflite") / 1e6)

qat_kd:  9.543926
quantized_state:  9.292454
TFlite: 9.026928


In [ ]:
print(quantized_student)

In [ ]:
quantized_student.print_readable()

In [ ]:
print(quantized_student.graph)